In [ ]:
# ============================================================
# 📦 IMPORTS
# Chargement des bibliothèques
# ============================================================

import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix

In [ ]:
# ============================================================
# 📌 Consignes de la quête
# 
# ============================================================

# 1.Importe cet ensemble de données de tweets dans un DataFrame.
# 2.Ne garde que les tweets positifs et négatifs (tu excluras donc les neutral). Quel est le pourcentage de tweets positifs/négatifs ?
# 3.Copie la colonne text dans une Série X, et la colonne sentiment dans une Série y. Applique un train test split avec le random_state = 32 et un train_size de 0.75.
# 4.Crée un modèle vectorizer avec scikit-learn en utilisant la méthode Countvectorizer. Entraîne ton modèle sur X_train, puis crée une matrice de features X_train_CV. Crée la matrice X_test_CV sans ré-entraîner le modèle. Le format de la matrice X_test_CV doit être 4091x15806 avec 44633 stored elements.
# 5.Entraîne maintenant une régression logistique avec les paramètres par défaut. Tu devrais obtenir les résultats suivants : 0.966pour le test d'entraînement, et 0.877 pour l'ensemble de test.
# 6.Étape bonus : essaye d'afficher 10 tweets qui ont été mal prédits (faux positifs ou faux négatifs). Aurais-tu fait mieux que l'algorithme ?

In [ ]:
# ============================================================
# 📂 CHARGEMENT DES DONNÉES
# Lecture et premier aperçu du dataset
# ============================================================

def read_csv(fichier):
    df = pd.read_csv(fichier)
    return df

df = read_csv("train.csv")

In [ ]:
# ============================================================
# 🔍 EXPLORATION / EDA
# Analyse exploratoire des données
# ============================================================

def quick_explore(dataframe):
    """Exploration rapide d'un DataFrame"""
    print('##### Observer les lignes #####')
    display(dataframe.head(10))
    print('\n')

    print('##### Dimensions du dataset #####')
    print(f'Lignes : {dataframe.shape[0]}, Colonnes : {dataframe.shape[1]}')
    print('\n')

    print('##### Informations sur les colonnes #####')
    print(dataframe.info())
    print('\n')

    print('##### Noms des colonnes #####')
    print(list(dataframe.columns))
    print('\n')

    print('##### Valeurs uniques #####')
    print(dataframe.nunique())
    print('\n')

    print('##### Description des colonnes numériques #####')
    stats = dataframe.describe(percentiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]).T
    print(stats.to_markdown())
    print('\n')

    print('##### Pourcentage de NaN par colonne #####')
    nan_percentage = (dataframe.isnull().sum() / len(dataframe)) * 100
    only_nan  = nan_percentage[nan_percentage > 0].sort_values(ascending=False)
    print(only_nan)
    print('\n')

    print('##### Nombre de doublons #####')
    print(dataframe.duplicated().sum(), 'doublon(s) trouvé(s)')
    print('\n')

    print('##### NaN par colonne #####')
    print(dataframe.isna().sum())

In [ ]:
# ============================================================
# 🔍 EXPLORATION / EDA
# Analyse exploratoire des données
# ============================================================

quick_explore(df)

In [ ]:
# ============================================================
# 📌 Suppression des tweets "neutral"
# Je supprime les tweets neutral et je garde positive et negative
# ============================================================

# Je crée une condition pour garder que les lignes sans neutral
condition = df['sentiment'] != 'neutral'
df_sans_neutral = df[condition]
display(df_sans_neutral['sentiment'].value_counts())

In [ ]:
# ============================================================
# 📌 Pourcentage des tweets positifs et negatifs
# 
# ============================================================

tweet_positif = df_sans_neutral['sentiment'] == 'positive'
tweet_negatif = df_sans_neutral['sentiment'] == 'negative'

pourcentage_positif = round((len(df_sans_neutral[tweet_positif]) / len(df_sans_neutral['sentiment']) * 100),2)
print(f"Le pourcentage de tweet positif est de {pourcentage_positif}%")

pourcentage_negatif = round((len(df_sans_neutral[tweet_negatif]) / len(df_sans_neutral['sentiment']) * 100),2)
print(f"Le pourcentage de tweet négatif est de {pourcentage_negatif}%")

In [ ]:
# ============================================================
# 📌 Je définis X et y
#  J'applique le train_test_split sur mes features et ma cible
# ============================================================

X = df_sans_neutral['text']
y = df_sans_neutral['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=32)

In [ ]:
# ============================================================
# 📌 Je crée un modèle CountVectorizer avec la methode CountVectorizer
#  J'entraine mon modèle sur X_train
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
vectorizer.fit(X_train)

In [ ]:
# ============================================================
# 📌 Je crée une matrice de features et ma matrice X_test_CV
#  J'entraine mon X_train et je crée mes matmrices
# ============================================================

X_train_CV = vectorizer.fit_transform(X_train)
X_test_CV = vectorizer.transform(X_test)

print(X_test_CV)

In [ ]:
# ============================================================
# 📌 J'entraine avec une LogisticRegression
# 
# ============================================================

def train_model(X_train, y_train, modele='logistic'):
    """
    Utilisation : model = train_model(X_train, y_train)
    Modeles disponibles : 'logistic', 'decision_tree', 'random_forest', 'linear', 'KNN'
    """
    if modele == 'logistic':
        model = LogisticRegression()
    elif modele == 'decision_tree':
        model = DecisionTreeClassifier()
    elif modele == 'random_forest':
        model = RandomForestClassifier()
    elif modele == 'linear':
        model = LinearRegression()
    elif modele == 'KNN':
        model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    return model

model = train_model(X_train_CV, y_train, modele='logistic')

In [ ]:
# ============================================================
# 📌 Le modèle est entrainé
# Maintenant je calcule les scores
# ============================================================

print(f"Le modele obtient un accuracy de {model.score(X_train_CV, y_train)} sur le train et {model.score(X_test_CV, y_test)} sur le test")

In [ ]:
# J'instancie y_pred

y_pred = model.predict(X_test_CV)

# Je vérifie l'ordre des labels pour ma matrice de confusion 

np.unique(y_test)

In [ ]:
# ============================================================
# 📌 J'affiche une matrice de confusion
# Pour me faire une première idée
# ============================================================

def matrice_confusion(y_test, y_pred, labels=["negative","positive"]):
    """
    Afficher la matrice de confusion
    Avec labels : matrice_confusion(y_test, y_pred, labels=['Non', 'Oui'])
    """
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predit')
    plt.ylabel('Reel')
    plt.title('Matrice de Confusion')
    plt.show()

matrice_confusion(y_test, y_pred)



In [ ]:
# ============================================================
# 📌 J'essaye d'afficher 10 tweets mal prédits
# 
# ============================================================

# Je prends tous les tweets du jeu de test
# où la prédiction EST DIFFÉRENTE de la vraie valeur
erreurs = X_test[y_pred != y_test]

# Je récupère les vraies étiquettes de ces tweets ratés
vrais = y_test[y_pred != y_test]

# Je récupère ce que le modèle a prédit pour ces tweets
predits = y_pred[y_pred != y_test]

# Je crée un tableau avec les 3 infos côte à côte
# pour voir : le tweet | ce qu'il était vraiment | ce que le modèle a cru
df_erreurs = pd.DataFrame({
    'tweet': erreurs.values,
    'reel': vrais.values,
    'predit': predits
})

# J'affiche les 10 premiers
display(df_erreurs.head(10))